# 🐼 KathaGemma: GPU-Accelerated Colab Runner

This notebook allows you to run the complete **KathaGemma** multi-agent backend on a free **T4 or L4 Google Colab GPU** for fast, low-latency Gemma-4 model execution, and tunnels the React UI directly to your local web browser.

### Step 1: Connect to GPU and Clone the Codebase

In [ ]:
# 1. Clone your consolidated repository (Replace with your actual GitHub URL if different)
!git clone https://github.com/DevByPawan/KathaGemma.git
%cd KathaGemma
!git checkout kathagemma_demo

### Step 2: Install Project Dependencies
This installs PyTorch (GPU enabled), PEFT adapters, ChromaDB vector stores, and Node utilities for tunneling.

In [ ]:
# Install python requirements
!pip install -r requirements.txt

!pip install -U torchao

# Install tutoring helpers
!pip install pyngrok nest-asyncio python-dotenv

# Install localtunnel globally via npm
!npm install -g localtunnel


### Step 3: Start the Backend & Tunnel
Run this cell to start the FastAPI server, expose it to a public URL using Localtunnel, and build the React UI dynamically pointing to this public address.

In [ ]:
import os
import time
import subprocess
import nest_asyncio
import urllib.request
nest_asyncio.apply()

# 1. Fetch public IP (needed as password for localtunnel reminder page bypass)
public_ip = ""
try:
    public_ip = urllib.request.urlopen('https://ident.me').read().decode('utf8').strip()
    print(f"🔑 Your LocalTunnel Password (Public IP): {public_ip}")
    print("If the localtunnel web page asks for a password/tunnel IP, paste this value in.\n")
except Exception:
    pass

# 2. Start LocalTunnel with custom subdomain 'kathagemma'
print("Starting LocalTunnel on port 8000...")
tunnel_process = subprocess.Popen(
    ["lt", "--port", "8000", "--subdomain", "kathagemma"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Give localtunnel time to connect
time.sleep(8)

# 3. Read the public tunnel URL from localtunnel stdout
public_url = ""
try:
    line = tunnel_process.stdout.readline().strip()
    if "https://" in line:
        public_url = line.split("your url is:")[-1].strip()
except Exception:
    pass

if not public_url:
    public_url = "https://kathagemma.loca.lt"
    print("Could not auto-detect tunnel URL. Using default: https://kathagemma.loca.lt")

print(f"\n🚀 SUCCESS! Your Backend Public Tunnel URL is: {public_url}\n")

# 4. Write Backend .env configuration
with open(".env", "w", encoding="utf-8") as f:
    f.write(f"BASE_URL={public_url}\n")
    f.write("USE_LOCAL_LORA=1\n")
    f.write("# GEMINI_API_KEY=your_key_here\n")

# 5. Write Frontend .env configuration
os.makedirs("KathaGemmaUI/frontend", exist_ok=True)
with open("KathaGemmaUI/frontend/.env", "w", encoding="utf-8") as f:
    f.write(f"VITE_API_URL={public_url}/api\n")

# 6. Build the React UI pointing to the new public tunnel API
print("Building the React frontend package...")
os.system("npm install --prefix KathaGemmaUI/frontend")
os.system("npm run build --prefix KathaGemmaUI/frontend")
print("Frontend build complete! Direct static serving is now enabled.")


### Step 4: Run the Server
Execute this cell to launch the FastAPI server. Because it serves the compiled React app at the root `/` route, clicking the **Public Tunnel URL** printed in Step 3 will load the complete app directly in your web browser!

In [ ]:
# 1. Load GEMINI_API_KEY from Colab Secrets or set it here manually
import os
try:
    from google.colab import userdata
    val = userdata.get('GEMINI_API_KEY')
    hf = userdata.get('HF_TOKEN')
    if val:
        os.environ['GEMINI_API_KEY'] = val
        print('✅ Loaded GEMINI_API_KEY from Colab Secrets successfully!')
    if hf:
        os.environ['HF_TOKEN'] = hf
        print('✅ Loaded HF_TOKEN from Colab Secrets successfully!')

except Exception as e:
    print('Could not read key from Colab Secrets. Checking local environment variable...')

# If not in Colab Secrets, manually uncomment and paste your key below:
# os.environ['GEMINI_API_KEY'] = 'your_actual_gemini_api_key_here'

# 2. Run the FastAPI server
!python app.py
